In [7]:
from pathlib import Path
import os
import sys
import json
import shutil
import logging
import platform
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name != "fruitnet-chili-yield":
    if PROJECT_ROOT.parent.name == "fruitnet-chili-yield":
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

assert PROJECT_ROOT.name == "fruitnet-chili-yield", (
    f"Please run this notebook from fruitnet-chili-yield root. Current: {PROJECT_ROOT}"
)

os.chdir(PROJECT_ROOT)

for d in [
    "logs",
    "models/fruitnet",
    "models/baselines",
    "configs/models",
    "configs/data",
    "configs/train",
    "configs/rl",
    "configs/deployment",
    "results/model_registry",
    "artifacts/pretrained",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

LOG_PATH = Path("logs") / f"00_init_models_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"


class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()


_log_fp = open(LOG_PATH, "a", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, _log_fp)
sys.stderr = Tee(sys.__stderr__, _log_fp)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(sys.__stdout__),
    ],
)

logging.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logging.info("Python=%s", sys.version)
logging.info("Platform=%s", platform.platform())
logging.info("Log file=%s", LOG_PATH)

In [8]:
coord_attention_py = r'''
import torch
import torch.nn as nn


class CoordAtt(nn.Module):
    """
    Coordinate Attention block with same input/output channels.

    YAML usage in Ultralytics:
        - [-1, 1, CoordAtt, [C]]

    The C value must match the previous layer output channel.
    """

    def __init__(self, channels: int, reduction: int = 32):
        super().__init__()

        self.channels = int(channels)
        mip = max(8, self.channels // reduction)

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        self.conv1 = nn.Conv2d(self.channels, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU(inplace=True)

        self.conv_h = nn.Conv2d(mip, self.channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, self.channels, kernel_size=1, stride=1, padding=0)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))

        return identity * a_h * a_w
'''

siou_loss_py = r'''
import torch


def bbox_iou_xyxy(box1, box2, eps=1e-7):
    x1 = torch.max(box1[..., 0], box2[..., 0])
    y1 = torch.max(box1[..., 1], box2[..., 1])
    x2 = torch.min(box1[..., 2], box2[..., 2])
    y2 = torch.min(box1[..., 3], box2[..., 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    area1 = (box1[..., 2] - box1[..., 0]).clamp(min=0) * (
        box1[..., 3] - box1[..., 1]
    ).clamp(min=0)

    area2 = (box2[..., 2] - box2[..., 0]).clamp(min=0) * (
        box2[..., 3] - box2[..., 1]
    ).clamp(min=0)

    return inter / (area1 + area2 - inter + eps)


def siou_loss_xyxy(pred, target, eps=1e-7):
    """
    Standalone SIoU-style loss.

    Important:
    This file defines SIoU math, but Ultralytics YOLO.train()
    will not automatically use it unless we later implement
    a custom trainer/loss hook.
    """

    iou = bbox_iou_xyxy(pred, target, eps=eps)

    px = (pred[..., 0] + pred[..., 2]) / 2
    py = (pred[..., 1] + pred[..., 3]) / 2
    gx = (target[..., 0] + target[..., 2]) / 2
    gy = (target[..., 1] + target[..., 3]) / 2

    pw = (pred[..., 2] - pred[..., 0]).clamp(min=eps)
    ph = (pred[..., 3] - pred[..., 1]).clamp(min=eps)
    gw = (target[..., 2] - target[..., 0]).clamp(min=eps)
    gh = (target[..., 3] - target[..., 1]).clamp(min=eps)

    cw = (
        torch.max(pred[..., 2], target[..., 2])
        - torch.min(pred[..., 0], target[..., 0])
    ).clamp(min=eps)

    ch = (
        torch.max(pred[..., 3], target[..., 3])
        - torch.min(pred[..., 1], target[..., 1])
    ).clamp(min=eps)

    dx = gx - px
    dy = gy - py
    sigma = torch.sqrt(dx**2 + dy**2 + eps)

    sin_alpha_1 = torch.abs(dx) / sigma
    sin_alpha_2 = torch.abs(dy) / sigma

    threshold = torch.sqrt(torch.tensor(2.0, device=pred.device)) / 2
    sin_alpha = torch.where(sin_alpha_1 > threshold, sin_alpha_2, sin_alpha_1)

    angle_cost = torch.cos(torch.arcsin(sin_alpha) * 2 - torch.pi / 2)

    rho_x = (dx / cw) ** 2
    rho_y = (dy / ch) ** 2

    gamma = 2 - angle_cost
    distance_cost = 2 - torch.exp(-gamma * rho_x) - torch.exp(-gamma * rho_y)

    omega_w = torch.abs(pw - gw) / torch.max(pw, gw)
    omega_h = torch.abs(ph - gh) / torch.max(ph, gh)

    shape_cost = (1 - torch.exp(-omega_w)) ** 4 + (
        1 - torch.exp(-omega_h)
    ) ** 4

    loss = 1 - iou + 0.5 * (distance_cost + shape_cost)

    return loss
'''

register_py = r'''
def register_fruitnet_modules():
    """
    Register custom FruitNet modules into Ultralytics parser namespace.
    Run this before calling YOLO("configs/models/fruitnet_gcs.yaml").
    """

    from models.fruitnet.coord_attention import CoordAtt
    import ultralytics.nn.tasks as tasks

    tasks.CoordAtt = CoordAtt

    try:
        import ultralytics.nn.modules as modules
        modules.CoordAtt = CoordAtt
    except Exception:
        pass

    return {"CoordAtt": CoordAtt}
'''

Path("models/fruitnet/coord_attention.py").write_text(
    coord_attention_py, encoding="utf-8"
)

Path("models/fruitnet/siou_loss.py").write_text(
    siou_loss_py, encoding="utf-8"
)

Path("models/fruitnet/register_ultralytics_modules.py").write_text(
    register_py, encoding="utf-8"
)

Path("models/fruitnet/__init__.py").write_text(
    "from .coord_attention import CoordAtt\n", encoding="utf-8"
)

logging.info("Wrote FruitNet Python modules.")

In [9]:
coord_attention_py = r'''
import torch
import torch.nn as nn


class CoordAtt(nn.Module):
    """
    Coordinate Attention block with same input/output channels.

    YAML usage in Ultralytics:
        - [-1, 1, CoordAtt, [C]]

    The C value must match the previous layer output channel.
    """

    def __init__(self, channels: int, reduction: int = 32):
        super().__init__()

        self.channels = int(channels)
        mip = max(8, self.channels // reduction)

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        self.conv1 = nn.Conv2d(self.channels, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU(inplace=True)

        self.conv_h = nn.Conv2d(mip, self.channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, self.channels, kernel_size=1, stride=1, padding=0)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))

        return identity * a_h * a_w
'''

siou_loss_py = r'''
import torch


def bbox_iou_xyxy(box1, box2, eps=1e-7):
    x1 = torch.max(box1[..., 0], box2[..., 0])
    y1 = torch.max(box1[..., 1], box2[..., 1])
    x2 = torch.min(box1[..., 2], box2[..., 2])
    y2 = torch.min(box1[..., 3], box2[..., 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    area1 = (box1[..., 2] - box1[..., 0]).clamp(min=0) * (
        box1[..., 3] - box1[..., 1]
    ).clamp(min=0)

    area2 = (box2[..., 2] - box2[..., 0]).clamp(min=0) * (
        box2[..., 3] - box2[..., 1]
    ).clamp(min=0)

    return inter / (area1 + area2 - inter + eps)


def siou_loss_xyxy(pred, target, eps=1e-7):
    """
    Standalone SIoU-style loss.

    Important:
    This file defines SIoU math, but Ultralytics YOLO.train()
    will not automatically use it unless we later implement
    a custom trainer/loss hook.
    """

    iou = bbox_iou_xyxy(pred, target, eps=eps)

    px = (pred[..., 0] + pred[..., 2]) / 2
    py = (pred[..., 1] + pred[..., 3]) / 2
    gx = (target[..., 0] + target[..., 2]) / 2
    gy = (target[..., 1] + target[..., 3]) / 2

    pw = (pred[..., 2] - pred[..., 0]).clamp(min=eps)
    ph = (pred[..., 3] - pred[..., 1]).clamp(min=eps)
    gw = (target[..., 2] - target[..., 0]).clamp(min=eps)
    gh = (target[..., 3] - target[..., 1]).clamp(min=eps)

    cw = (
        torch.max(pred[..., 2], target[..., 2])
        - torch.min(pred[..., 0], target[..., 0])
    ).clamp(min=eps)

    ch = (
        torch.max(pred[..., 3], target[..., 3])
        - torch.min(pred[..., 1], target[..., 1])
    ).clamp(min=eps)

    dx = gx - px
    dy = gy - py
    sigma = torch.sqrt(dx**2 + dy**2 + eps)

    sin_alpha_1 = torch.abs(dx) / sigma
    sin_alpha_2 = torch.abs(dy) / sigma

    threshold = torch.sqrt(torch.tensor(2.0, device=pred.device)) / 2
    sin_alpha = torch.where(sin_alpha_1 > threshold, sin_alpha_2, sin_alpha_1)

    angle_cost = torch.cos(torch.arcsin(sin_alpha) * 2 - torch.pi / 2)

    rho_x = (dx / cw) ** 2
    rho_y = (dy / ch) ** 2

    gamma = 2 - angle_cost
    distance_cost = 2 - torch.exp(-gamma * rho_x) - torch.exp(-gamma * rho_y)

    omega_w = torch.abs(pw - gw) / torch.max(pw, gw)
    omega_h = torch.abs(ph - gh) / torch.max(ph, gh)

    shape_cost = (1 - torch.exp(-omega_w)) ** 4 + (
        1 - torch.exp(-omega_h)
    ) ** 4

    loss = 1 - iou + 0.5 * (distance_cost + shape_cost)

    return loss
'''

register_py = r'''
def register_fruitnet_modules():
    """
    Register custom FruitNet modules into Ultralytics parser namespace.
    Run this before calling YOLO("configs/models/fruitnet_gcs.yaml").
    """

    from models.fruitnet.coord_attention import CoordAtt
    import ultralytics.nn.tasks as tasks

    tasks.CoordAtt = CoordAtt

    try:
        import ultralytics.nn.modules as modules
        modules.CoordAtt = CoordAtt
    except Exception:
        pass

    return {"CoordAtt": CoordAtt}
'''

Path("models/fruitnet/coord_attention.py").write_text(
    coord_attention_py, encoding="utf-8"
)

Path("models/fruitnet/siou_loss.py").write_text(
    siou_loss_py, encoding="utf-8"
)

Path("models/fruitnet/register_ultralytics_modules.py").write_text(
    register_py, encoding="utf-8"
)

Path("models/fruitnet/__init__.py").write_text(
    "from .coord_attention import CoordAtt\n", encoding="utf-8"
)

logging.info("Wrote FruitNet Python modules.")

In [10]:
MODEL_REGISTRY = {
    "yolov5s": {
        "description": "YOLOv5 small baseline through Ultralytics unified weights.",
        "model_uri": "yolov5su.pt",
        "role_table_4_1": True,
        "custom": False,
    },
    "yolov8n": {
        "description": "YOLOv8 nano baseline.",
        "model_uri": "yolov8n.pt",
        "role_table_4_1": True,
        "custom": False,
    },
    "fruitnet_b": {
        "description": "Internal baseline: compact YOLOv8-style detector without GhostConv, Coordinate Attention, or SIoU.",
        "model_uri": "configs/models/fruitnet_b.yaml",
        "role_table_4_1": True,
        "custom": True,
    },
    "fruitnet_g": {
        "description": "FruitNet-B + GhostConv.",
        "model_uri": "configs/models/fruitnet_g.yaml",
        "role_ablation": True,
        "custom": True,
    },
    "fruitnet_gc": {
        "description": "FruitNet-G + Coordinate Attention.",
        "model_uri": "configs/models/fruitnet_gc.yaml",
        "role_ablation": True,
        "custom": True,
    },
    "fruitnet_gcs": {
        "description": "FruitNet-GC + SIoU math prepared. SIoU needs custom trainer hook later.",
        "model_uri": "configs/models/fruitnet_gcs.yaml",
        "role_table_4_1": True,
        "custom": True,
    },
}

Path("results/model_registry/model_registry.json").write_text(
    json.dumps(MODEL_REGISTRY, indent=2), encoding="utf-8"
)

print(json.dumps(MODEL_REGISTRY, indent=2))
logging.info("Saved model registry.")

In [11]:
fruitnet_b_yaml = """
# FruitNet-B
# YOLOv8-tiny-like internal baseline.
# No GhostConv, no Coordinate Attention, no SIoU custom trainer.

nc: 80

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C2f, [512]]

  - [[15, 18, 21], 1, Detect, [nc]]
"""

fruitnet_g_yaml = """
# FruitNet-G
# FruitNet-B + GhostConv in major downsampling blocks.

nc: 80

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, GhostConv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, GhostConv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, GhostConv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [-1, 2, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C2f, [128]]

  - [-1, 1, GhostConv, [128, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]

  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C2f, [512]]

  - [[15, 18, 21], 1, Detect, [nc]]
"""

fruitnet_gc_yaml = """
# FruitNet-GC
# FruitNet-G + Coordinate Attention.

nc: 80

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, GhostConv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, CoordAtt, [64]]

  - [-1, 1, GhostConv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, CoordAtt, [128]]

  - [-1, 1, GhostConv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, GhostConv, [512, 3, 2]]
  - [-1, 2, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 1, CoordAtt, [512]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C2f, [128]]
  - [-1, 1, CoordAtt, [128]]

  - [-1, 1, GhostConv, [128, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C2f, [512]]
  - [-1, 1, CoordAtt, [512]]

  - [[21, 25, 29], 1, Detect, [nc]]
"""

fruitnet_gcs_yaml = """
# FruitNet-GCS
# GhostConv + Coordinate Attention.
# SIoU math is written to models/fruitnet/siou_loss.py.
# Full SIoU activation requires later custom trainer/loss hook.

nc: 80

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, GhostConv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, CoordAtt, [64]]

  - [-1, 1, GhostConv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, CoordAtt, [128]]

  - [-1, 1, GhostConv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, GhostConv, [512, 3, 2]]
  - [-1, 2, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 1, CoordAtt, [512]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C2f, [128]]
  - [-1, 1, CoordAtt, [128]]

  - [-1, 1, GhostConv, [128, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]
  - [-1, 2, C2f, [256]]
  - [-1, 1, CoordAtt, [256]]

  - [-1, 1, GhostConv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C2f, [512]]
  - [-1, 1, CoordAtt, [512]]

  - [[21, 25, 29], 1, Detect, [nc]]
"""

yamls = {
    "fruitnet_b": fruitnet_b_yaml,
    "fruitnet_g": fruitnet_g_yaml,
    "fruitnet_gc": fruitnet_gc_yaml,
    "fruitnet_gcs": fruitnet_gcs_yaml,
}

for name, text in yamls.items():
    path = Path("/home/diy-hus/fruitnet-chili-yield/models") / f"{name}.yaml"
    path.write_text(text.strip() + "\n", encoding="utf-8")
    logging.info("Wrote %s", path)